# One-dimensional heat conduction with finite differences

Copyright © 2026 Philip Eisenlohr and contributors. [License](https://github.com/mseMSU/Notebooks-pub/blob/main/LICENSE.md)

In [ ]:
# Load the shared notebook utilities locally or, in Colab, from GitHub.
try:
    import nbkit
except ModuleNotFoundError:
    from pathlib import Path
    from urllib.request import urlretrieve
    import sys
    import tempfile

    module_path = next(
        (candidate
         for parent in (Path.cwd(), *Path.cwd().parents)
         for candidate in (parent / 'nbkit.py', parent / 'notebooks' / 'nbkit.py')
         if candidate.is_file()),
        None,
    )

    if module_path is None:
        support_dir = Path(tempfile.gettempdir()) / 'notebook_tools'
        support_dir.mkdir(exist_ok=True)
        module_path = support_dir / 'nbkit.py'
        urlretrieve(
            'https://raw.githubusercontent.com/'
            'mseMSU/Notebooks-pub/main/notebooks/nbkit.py',
            module_path,
        )

    sys.path.insert(0, str(module_path.parent))
    import nbkit


## Overview

This starter notebook illustrates how a finite-difference scheme turns a one-dimensional partial differential equation into repeated array updates.
We model a uniform rod whose ends are held at a reference temperature and watch an initially warm interior cool by heat conduction.
The example is intentionally minimal and can be expanded into a fuller lesson later.

After working through it, you should be able to identify a spatial grid, a time step, boundary conditions, and the stability limit of an explicit scheme.

## Model and numerical scheme

Let $\theta(x,t)=T(x,t)-T_{\mathrm{ref}}$ denote the temperature excess above the fixed reference temperature at the rod ends.
For constant thermal diffusivity $\alpha$ and no internal heat generation, the heat equation is

$$
\frac{\partial \theta}{\partial t}=\alpha\frac{\partial^2\theta}{\partial x^2}, \qquad 0<x<L.
$$

We impose $\theta(0,t)=\theta(L,t)=0$ and choose the initial profile $\theta(x,0)=A\sin(\pi x/L)$.
Here, $L$ is measured in metres, time in seconds, $\alpha$ in square metres per second, and $\theta$ in kelvin.

Using a forward difference in time and a centered difference in space gives

$$
\theta_i^{n+1}=\theta_i^n+r\left(\theta_{i+1}^n-2\theta_i^n+\theta_{i-1}^n\right), \qquad r=\frac{\alpha\Delta t}{\Delta x^2}.
$$

The index $i$ labels position and $n$ labels time.
For this constant-coefficient, one-dimensional explicit scheme on a uniform grid, stability requires $r\leq 1/2$.
Every interior value must be updated from the same old time level; the boundary values remain fixed.

Before revealing or running the outputs, predict how the peak height and the shape of the temperature profile will change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L = 0.1                    # Rod length, m
alpha = 1.0e-4              # Thermal diffusivity, m^2/s
amplitude = 100.0           # Initial peak temperature excess, K
nx = 51                    # Grid points, including the two boundaries
t_final = 5.0               # Final time, s
target_r = 0.4              # Safely below the explicit stability limit

x = np.linspace(0.0, L, nx)
dx = x[1] - x[0]
nsteps = int(np.ceil(t_final / (target_r * dx**2 / alpha)))
dt = t_final / nsteps       # Adjust slightly to land exactly at t_final
r = alpha * dt / dx**2
assert 0.0 < r <= 0.5

theta = amplitude * np.sin(np.pi * x / L)
theta[[0, -1]] = 0.0
snapshots = [(0.0, theta.copy())]
save_steps = set(np.linspace(0, nsteps, 5, dtype=int)[1:])

for step in range(1, nsteps + 1):
    old = theta.copy()
    theta[1:-1] = old[1:-1] + r * (
        old[2:] - 2.0 * old[1:-1] + old[:-2]
    )
    theta[[0, -1]] = 0.0
    if step in save_steps:
        snapshots.append((step * dt, theta.copy()))

print(f"dx = {dx:.4f} m, dt = {dt:.5f} s, r = {r:.3f}")
print(f"Time steps: {nsteps}")
print(f"Final peak temperature excess: {theta.max():.2f} K")


## Compare with a known solution

For this particular initial profile and these boundary conditions, the exact solution is

$$
\theta(x,t)=A\sin(\pi x/L)\exp\left[-\alpha(\pi/L)^2t\right].
$$

Its shape stays sinusoidal while its amplitude decays.
We compare this expression with the numerical solution at the final time.
Agreement here provides a useful check, although it is not a proof that every possible input or implementation is correct.

In [ ]:
exact = amplitude * np.sin(np.pi * x / L) * np.exp(
    -alpha * (np.pi / L)**2 * t_final
)
error = np.max(np.abs(theta - exact))
print(f"Maximum absolute error at t = {t_final:g} s: {error:.4f} K")

fig, ax = plt.subplots()
for time, profile in snapshots:
    ax.plot(x, profile, label=f"t = {time:.2f} s")
ax.plot(x, exact, "k--", label="Exact solution at final time")
ax.set_xlabel("Position (m)")
ax.set_ylabel("Temperature excess (K)")
ax.set_title("Cooling of a rod with fixed-temperature ends")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Try next

1. Increase the thermal diffusivity and predict how much faster the rod cools before running the notebook again.
2. Refine the grid by increasing `nx`, keeping `target_r` fixed, and compare the final error and number of time steps.
3. Replace the initial profile with another smooth shape that is zero at both ends.
   The displayed exact solution applies only to the original sine profile, so remove that comparison for other initial conditions.

Future extensions could introduce boundary heat fluxes, internal heat generation, implicit time stepping, and a systematic convergence study.
